In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import normalized_mutual_info_score
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import roc_auc_score, auc
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, average_precision_score
from sklearn.svm import OneClassSVM
from sklearn.ensemble import HistGradientBoostingClassifier

np.random.seed(560)
SEED=560 #random seed

from google.colab import drive
import os
import joblib
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
model = joblib.load("/content/drive/MyDrive/ml_project/model.pkl")
num_imputer = joblib.load("/content/drive/MyDrive/ml_project/num_imputer.pkl")
scaler = joblib.load("/content/drive/MyDrive/ml_project/scaler.pkl")
cat_imputer = joblib.load("/content/drive/MyDrive/ml_project/cat_imputer.pkl")
train_columns = joblib.load("/content/drive/MyDrive/ml_project/columns.pkl")
best_threshold = joblib.load("/content/drive/MyDrive/ml_project/threshold.pkl")

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving balanced_500k.log.gz to balanced_500k.log (5).gz


In [ ]:
df = pd.read_csv(
    "balanced_500k.log.gz",
    sep=r"\s+",
    comment="#",
    header=None,
    engine="python"
)

df.columns = [
    "ts", "uid", "id.orig_h", "id.orig_p", "id.resp_h", "id.resp_p",
    "proto", "service", "duration", "orig_bytes", "resp_bytes",
    "conn_state", "local_orig", "local_resp", "missed_bytes",
    "history", "orig_pkts", "orig_ip_bytes", "resp_pkts",
    "resp_ip_bytes", "tunnel_parents", "label", "detailed_label"
]

In [ ]:
df["label"] = df["label"].str.strip().str.replace("-", "")
df = df.drop(columns=[
    "uid",
    "id.orig_h",
    "id.resp_h",
    "tunnel_parents",
    "detailed_label",
    "ts",
    "id.orig_p"
], errors="ignore")


In [ ]:
X_sub = df.sample(n=100000, random_state=24)
y_true = X_sub["label"].apply(lambda x: 0 if x == "Benign" else 1)
X = X_sub.drop(columns=['label'])

In [ ]:
numeric = X.select_dtypes(include=["number"]).columns.tolist()
categorical = X.select_dtypes(include=["object"]).columns.tolist()

X[numeric] = num_imputer.transform(X[numeric])
X[categorical] = cat_imputer.transform(X[categorical])
X[numeric] = scaler.transform(X[numeric])
X = pd.get_dummies(X, columns=categorical, drop_first=True)
X = X.reindex(columns=train_columns, fill_value=0)

In [ ]:
probs = model.predict_proba(X)[:, 1]
preds = (probs >= best_threshold).astype(int)

In [ ]:
print("STATISTICS:")
print("Accuracy: ", accuracy_score(y_true, preds))
print("Precision: ", precision_score(y_true, preds))
print("Recall: ", recall_score(y_true, preds))
print("F1: ", f1_score(y_true, preds))
print("PR_AUC: ", average_precision_score(y_true, probs))
print("ROC_AUC: ", roc_auc_score(y_true, probs))

STATISTICS:
Accuracy:  0.99228
Precision:  0.9985600129801647
Recall:  0.9859620314001922
F1:  0.9922210354487011
PR_AUC:  0.9944030007558795
ROC_AUC:  0.9948615479811602
